In [ ]:
# Kaggle 한 셀: GPU 0 = 원본 기준선 / GPU 1 = MTF 증강
# 새 세션에서 이전 Notebook Output을 Add Input으로 연결하면 latest부터 자동 재개합니다.
RUN_TAG = 'mtf_pair'            # 조건을 바꿔 새로 시작할 때 이름 변경
SEED = 46
K = 6
TOTAL_EPOCHS = 100
EPOCHS_THIS_SESSION = 100
MAX_HOURS = 9.0                 # 출력 보관 시간을 남기고 안전하게 저장/종료
EFFECTIVE_BATCH = 32
MICRO_BATCH = 16                # 두 모델에 동일 적용. OOM이면 둘 다 8로 새 실험 시작
AMP = True                     # T4 FP16 + GradScaler, validation은 FP32
GPU_RESIDENT = True             # QB 데이터·증강을 GPU에 상주시켜 CPU/H2D 병목 감소
WORKERS_PER_GPU = 2             # GPU 메모리가 부족할 때만 CPU fallback에 사용
DETERMINISTIC = False           # cuDNN autotune 사용; 비트 단위 재현은 보장하지 않음
CHECKPOINT_STEPS = 200          # micro batch 기준, optimizer step 경계에서 저장
RESUME_INPUT = ''               # 자동 발견이 여러 개면 이전 출력의 최상위 폴더 지정
SMOKE = False                   # True = 32패치/1에폭, 별도 결과 폴더

import os, sys, json, time, math, hashlib, shutil, subprocess, signal, zipfile, importlib.util
from pathlib import Path
from IPython.display import display, HTML, FileLink

# 필요한 작은 패키지만 설치하고 Kaggle의 torch/CUDA는 유지합니다.
missing = [name for name in ('h5py','numpy') if importlib.util.find_spec(name) is None]
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import h5py, numpy as np, torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2를 선택하세요.'
assert K in (4,6) and SEED >= 0 and RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
assert EFFECTIVE_BATCH % MICRO_BATCH == 0 and 1 <= MICRO_BATCH <= EFFECTIVE_BATCH
assert CHECKPOINT_STEPS % (EFFECTIVE_BATCH // MICRO_BATCH) == 0
assert TOTAL_EPOCHS >= 1 and EPOCHS_THIS_SESSION >= 1 and MAX_HOURS > 0
print('GPU:', [torch.cuda.get_device_name(i) for i in (0,1)])

# 아래 두 스냅샷은 파일 생성 시 삽입됩니다.
EMBEDDED_SOURCES = {'SSA-MRN/src/ssamrn/models/ssa_mrn.py': '"""Minimal forward-path repair around the untouched official network.py.\n\nThe official forward(pan, lms) references an undefined `ms`. Its three\nspatial stages require PAN and interpolated LMS at full resolution, plus\nthe original MS at one-quarter resolution. This adapter supplies that input\nexplicitly and retains the original modules, parameter names and operations.\n"""\n\nimport importlib.util\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\n\n\nUPSTREAM_PATH = Path(__file__).resolve().parents[3] / "references/upstream/network.py"\n\n\ndef _upstream_model():\n    if not UPSTREAM_PATH.is_file():\n        raise FileNotFoundError(f"Initialize the official SSA-MRN submodule: {UPSTREAM_PATH}")\n    spec = importlib.util.spec_from_file_location("ssamrn_official_network", UPSTREAM_PATH)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module.PansharpeningNet\n\n\nclass DirectMLPReLU(nn.PReLU):\n    """Equivalent PReLU expression avoiding a DirectML batch backward error."""\n\n    def forward(self, input):\n        slope = self.weight.view(1, -1, 1, 1)\n        return torch.where(input > 0, input, slope * input)\n\n\nclass RestoredPansharpeningNet(_upstream_model()):\n    """Official layers with an explicit low-resolution MS input."""\n\n    def __init__(self, channels, ssai_dimension=4, guide_channels=1):\n        super().__init__(channels)\n        if guide_channels not in (1, 3):\n            raise ValueError("guide_channels must be 1 (PAN) or 3 (RGB)")\n        self.guide_channels = guide_channels\n        if guide_channels == 3:\n            self.conv1t1 = nn.Conv2d(3, 3, 3, padding=1)\n            self.cov2t64 = nn.Conv2d(channels + 6, 64, 3, padding=1)\n            for blocks in (self.SSA_blocks, self.SSA_blocks1, self.SSA_blocks2):\n                for block in blocks:\n                    block.conv1t6 = nn.Conv2d(3, 4, 3, padding=1)\n        if ssai_dimension < 1:\n            raise ValueError("ssai_dimension must be positive")\n        self.ssai_dimension = ssai_dimension\n        if ssai_dimension == 4:\n            return  # Preserve the published upstream architecture and existing checkpoints.\n\n        # The upstream code hard-codes K=4, while the paper specifies K=6.\n        # Resize only the layers used by the restored forward path; leave upstream untouched.\n        self.fixed = ssai_dimension\n        for blocks in (self.SSA_blocks, self.SSA_blocks1, self.SSA_blocks2):\n            for block in blocks:\n                block.fixed = ssai_dimension\n                block.conv1t6 = nn.Conv2d(guide_channels, ssai_dimension, 3, padding=1)\n                block.conv6t6 = nn.Conv2d(ssai_dimension, ssai_dimension, 3, padding=1)\n                block.conv7t6_3 = nn.Conv2d(channels + 1, ssai_dimension, 3, padding=1)\n        for name in ("conv48tnum1", "conv48tnum2", "conv48tnum3"):\n            setattr(self, name, nn.Conv2d(ssai_dimension * channels, channels, 3, padding=1))\n\n    def use_directml_prelu(self):\n        """Keep PReLU weights/checkpoints while using a DirectML-safe backward."""\n        for name, module in self.named_modules():\n            if not name or not isinstance(module, nn.PReLU):\n                continue\n            parent = self.get_submodule(name.rpartition(".")[0]) if "." in name else self\n            replacement = DirectMLPReLU(module.num_parameters)\n            with torch.no_grad():\n                replacement.weight.copy_(module.weight)\n            setattr(parent, name.rpartition(".")[2], replacement)\n\n    def _attend(self, pan, features, ms, blocks, fusion, activation):\n        outputs = []\n        for band in range(self.channels):\n            band_input = torch.cat([features, ms[:, band : band + 1]], dim=1)\n            outputs.append(blocks[band](pan, band_input))\n        return activation(fusion(torch.cat(outputs, dim=1)))\n\n    def forward(self, pan, lms, ms):\n        if pan.ndim != 4 or lms.ndim != 4 or ms.ndim != 4:\n            raise ValueError("Expected NCHW tensors for pan, lms, and ms")\n        n, channels, h, w = lms.shape\n        if channels != self.channels or h % 4 or w % 4 or min(h, w) < 4:\n            raise ValueError(f"Invalid LMS shape: {tuple(lms.shape)}")\n        if pan.shape != (n, self.guide_channels, h, w) or ms.shape != (n, channels, h // 4, w // 4):\n            raise ValueError("Expected guide at LMS resolution and MS at 1/4 resolution")\n\n        pan_down_up = self.upsample1(self.downsample1(pan))\n        pan_updown = self.prelu(self.conv1t1(pan_down_up))\n        features = torch.cat([lms, pan, pan_updown], dim=1)\n        features = self.backbone(self.cov2t64(features))\n        res = self.conv64t48(features) + lms\n\n        pan_ms = self._attend(pan, res, lms, self.SSA_blocks, self.conv48tnum1, self.prelu1)\n        res = res + pan_ms\n\n        res_half = self.downsample100(res)\n        ms_half = self.upsample100(ms)\n        pan_half = self.downsample100(pan)\n        pan_ms = self._attend(pan_half, res_half, ms_half, self.SSA_blocks1, self.conv48tnum2, self.prelu2)\n        res_half = res_half + pan_ms\n\n        res_quarter = self.downsample200(res_half)\n        pan_quarter = self.downsample2(pan)\n        pan_ms = self._attend(pan_quarter, res_quarter, ms, self.SSA_blocks2, self.conv48tnum3, self.prelu3)\n\n        output = self.conv2ctc1(torch.cat([ms, pan_ms], dim=1))\n        output = self.conv2ctc2(torch.cat([self.upsample101(output), res_half], dim=1))\n        output = self.conv2ctc3(torch.cat([self.upsample102(output), res], dim=1))\n        return output\n', 'SSA-MRN/src/ssamrn/models/interp23.py': '"""Differentiable channelwise toolbox-style 23-tap LMS interpolation."""\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef interp23tap(image, ratio=4):\n    if image.ndim != 4 or ratio < 1 or ratio & (ratio-1):\n        raise ValueError(\'Expected NCHW input and power-of-two interpolation ratio\')\n    # Same coefficients, periodic boundary and insertion phases as metrics._interp23tap.\n    half = [.5,.305334091185,0,-.072698593239,0,.021809577942,\n            0,-.005192756653,0,.000807762146,0,-.000060081482]\n    with torch.autocast(image.device.type, enabled=False):\n        image=image.float()\n        kernel=image.new_tensor(half[:0:-1]+half)*2\n        channels=image.shape[1]\n        vertical=kernel.view(1,1,23,1).expand(channels,1,23,1).contiguous()\n        horizontal=kernel.view(1,1,1,23).expand(channels,1,1,23).contiguous()\n        for stage in range(int(math.log2(ratio))):\n            n,c,h,w=image.shape\n            up=image.new_zeros(n,c,h*2,w*2)\n            offset=1 if stage==0 else 0\n            up[:,:,offset::2,offset::2]=image\n            rows=torch.arange(-11,h*2+11,device=image.device)%(h*2)\n            cols=torch.arange(-11,w*2+11,device=image.device)%(w*2)\n            image=F.conv2d(up.index_select(2,rows),vertical,groups=c)\n            image=F.conv2d(image.index_select(3,cols),horizontal,groups=c)\n    return image\n\n\nclass Interp23(nn.Module):\n    def __init__(self,ratio):\n        super().__init__();self.ratio=ratio\n    def forward(self,x):\n        return interp23tap(x,self.ratio)\n', 'SSA-MRN/src/ssamrn/observation.py': '"""Differentiable RR MS observation using the MATLAB DLPan MTF protocol.\n\nDLPan-Toolbox at a34f884af88967580e3ef4f61a89e12fdfe4cda5:\nTools/genMTF.m, MTF.m, resize_images.m. Radial Kaiser window follows\nMathWorks fwind1\'s documented Huang construction. Training-only audits\nverify sampling and patch interiors; this does not establish FR sensor physics.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nSOURCE_COMMIT = "a34f884af88967580e3ef4f61a89e12fdfe4cda5"\nGNYQ = {\n    "QB": [0.34, 0.32, 0.30, 0.22],\n    "WV3": [0.325, 0.355, 0.360, 0.350, 0.365, 0.360, 0.335, 0.315],\n}\n\n\ndef module_hash():\n    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\n\ndef mtf_kernels(sensor, ratio=4, size=41):\n    """Return MATLAB-protocol real FIR kernels (C,1,N,N), including signed taps."""\n    if sensor not in GNYQ:\n        raise ValueError(f"No sourced GNyq coefficients for {sensor}; do not assume GF2=IKONOS")\n    if ratio != 4 or size != 41:\n        raise ValueError("Only audited x4, 41-tap kernels supported")\n    coordinate = np.arange(size) - (size - 1) / 2\n    y, x = np.meshgrid(coordinate, coordinate, indexing="ij")\n    radial_window = np.interp(np.sqrt(x*x+y*y), np.arange(size//2+1),\n                              np.kaiser(size, 0.5)[size//2:], right=0.0)\n    kernels = []\n    for gain in GNYQ[sensor]:\n        alpha = np.sqrt(((size - 1) / (2 * ratio)) ** 2 / (-2 * np.log(gain)))\n        desired = np.exp(-(x*x+y*y)/(2*alpha*alpha))\n        desired[desired < np.finfo(desired.dtype).eps*desired.max()] = 0\n        desired /= desired.max()\n        # MATLAB fwind1: inverse frequency response times radial 1-D window.\n        # Keep signed taps and DC gain; clipping/renormalizing changes the operator.\n        impulse = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(desired))) * radial_window\n        kernels.append(np.real(impulse))\n    return np.stack(kernels)[:, None].astype(np.float64)\n\n\nclass SensorObservation(nn.Module):\n    def __init__(self, sensor, sampling="decimate", phase=(2, 2), ratio=4):\n        super().__init__()\n        if sampling not in ("decimate", "bicubic"):\n            raise ValueError("Unknown sampling")\n        if len(phase) != 2 or any(v not in range(ratio) for v in phase):\n            raise ValueError("Invalid sampling phase")\n        self.sensor, self.sampling, self.phase, self.ratio = sensor, sampling, tuple(phase), ratio\n        self.register_buffer("kernel", torch.from_numpy(mtf_kernels(sensor, ratio)))\n\n    def blur(self, prediction):\n        if prediction.ndim != 4 or prediction.shape[1] != self.kernel.shape[0]:\n            raise ValueError("Expected NCHW with the sensor\'s bands")\n        if any(n % self.ratio for n in prediction.shape[-2:]):\n            raise ValueError("Spatial dimensions must be divisible by ratio")\n        # Replication implemented using index_select: deterministic CUDA backward,\n        # unlike replication_pad2d backward on some PyTorch versions.\n        padding = self.kernel.shape[-1] // 2\n        rows = torch.arange(-padding, prediction.shape[-2]+padding, device=prediction.device).clamp(0,prediction.shape[-2]-1)\n        cols = torch.arange(-padding, prediction.shape[-1]+padding, device=prediction.device).clamp(0,prediction.shape[-1]-1)\n        padded = prediction.index_select(-2, rows).index_select(-1, cols)\n        return F.conv2d(padded, self.kernel.to(dtype=prediction.dtype), groups=prediction.shape[1])\n\n    def sample(self, blurred):\n        if self.sampling == "bicubic":\n            return F.interpolate(blurred, scale_factor=1/self.ratio, mode="bicubic", align_corners=False, antialias=False)\n        return blurred[..., self.phase[0]::self.ratio, self.phase[1]::self.ratio]\n\n    def forward(self, prediction):\n        return self.sample(self.blur(prediction))\n\n    def loss(self, prediction, ms, margin, phases=None):\n        if phases is None:\n            observed = self(prediction)\n        else:\n            blurred = self.blur(prediction)\n            if len(phases) != len(prediction):\n                raise ValueError("Phase batch mismatch")\n            # Phases are fixed by original TRAIN pairs, never inferred from predictions.\n            observed = torch.cat([blurred[i:i+1, :, int(p[0])::self.ratio, int(p[1])::self.ratio]\n                                  for i, p in enumerate(phases)], dim=0)\n        if observed.shape != ms.shape:\n            raise ValueError("Observed prediction and MS shape differ")\n        if margin < 0 or min(ms.shape[-2:]) <= 2*margin:\n            raise ValueError("Invalid audited LR interior margin")\n        if margin:\n            observed = observed[..., margin:-margin, margin:-margin]\n            ms = ms[..., margin:-margin, margin:-margin]\n        return F.mse_loss(observed, ms)\n\n\ndef load_profile(path, expected_sha256=None, sensor=None, train_path=None):\n    path = Path(path)\n    content = path.read_bytes()\n    if expected_sha256 and hashlib.sha256(content).hexdigest() != expected_sha256:\n        raise ValueError("Observation profile hash changed")\n    profile = json.loads(content)\n    if profile.get("status") != "validated":\n        raise ValueError("Observation data audit has not passed")\n    if profile.get("operator_sha256") != module_hash():\n        raise ValueError("Observation code changed since audit")\n    if sensor and profile["sensor"] != sensor:\n        raise ValueError("Observation profile sensor mismatch")\n    if train_path and Path(profile["train_path"]).resolve() != Path(train_path).resolve():\n        raise ValueError("Observation profile dataset mismatch")\n    stat = Path(profile["train_path"]).stat()\n    if stat.st_size != profile["train_stat"]["size"] or stat.st_mtime_ns != profile["train_stat"]["mtime_ns"]:\n        raise ValueError("Training file changed since observation audit")\n    if profile.get("phase_policy") != "fixed_per_training_sample" or len(profile.get("phases",[])) != profile["sample_count"] or profile.get("coverage") != profile["sample_count"]:\n        raise ValueError("Training phase manifest incomplete")\n    if any(list(p) not in [[2,2],[1,2],[2,1]] for p in profile["phases"]):\n        raise ValueError("Unverified phase in manifest")\n    operator = SensorObservation(profile["sensor"],profile["sampling"],ratio=profile["ratio"])\n    return operator, profile\n', 'SSA-MRN/references/upstream/network.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass ResBlock(nn.Module):\n    def __init__(self, in_channels):\n        super(ResBlock, self).__init__()\n        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(in_channels)\n        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(in_channels)\n\n    def forward(self, x):\n        residual = x\n        out = self.conv1(x)\n        out = F.relu(self.bn1(out))\n        out = self.conv2(out)\n        out = self.bn2(out)\n        out += residual\n        out = F.relu(out)\n        return out\n\n\nclass Resbackbone(nn.Module):\n    def __init__(self, in_channels):\n        super(Resbackbone, self).__init__()\n        channel = 64\n        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=channel, kernel_size=3, padding=1, stride=1,\n                               bias=True)\n\n        self.res1 = ResBlock(channel)\n        self.res2 = ResBlock(channel)\n        self.res3 = ResBlock(channel)\n        self.res4 = ResBlock(channel)\n\n        self.backbone = nn.Sequential(\n            self.res1,\n            self.res2,\n            self.res3,\n            self.res4,\n\n        )\n        self.relu = nn.ReLU(inplace=True)\n        self.conv2 = nn.Conv2d(in_channels=channel, out_channels=1, stride=1, kernel_size=3, padding=1,\n                               bias=True)\n\n    def forward(self, x):\n        x = self.relu(self.conv1(x))\n        x = self.backbone(x)\n        x = self.conv2(x)\n        return x\n\n\nclass Resblock(nn.Module):\n    def __init__(self):\n        super(Resblock, self).__init__()\n\n        channel = 64\n\n        self.conv20 = nn.Conv2d(in_channels=channel, out_channels=channel, padding=1, kernel_size=3, stride=1,\n                                bias=True)\n        self.conv21 = nn.Conv2d(in_channels=channel, out_channels=channel, padding=1, kernel_size=3, stride=1,\n                                bias=True)\n        self.relu = nn.ReLU(\n            inplace=True) \n    def forward(self, x):\n        rs1 = self.relu(self.conv20(x))\n        rs2 = self.conv21(rs1)  \n        # print(f"rs2:{rs2.shape},x:{x.shape}")\n        rs = torch.add(x, rs2) \n        return rs\n\n\nclass SSA(nn.Module):\n    def __init__(self,channels):\n        super(SSA, self).__init__()\n\n        self.fixed  = 4\n\n\n        self.conv1t6 = nn.Conv2d(in_channels=1, out_channels=self.fixed , kernel_size=3, stride=1, padding=1,\n                                 bias=True)\n        self.conv6t12 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=3, stride=1, padding=1,\n                                  bias=True)\n        self.conv6t6 = nn.Conv2d(in_channels=self.fixed , out_channels=self.fixed , kernel_size=3, stride=1, padding=1,\n                                 bias=True)\n        self.relu = nn.ReLU()\n        self.conv7t6_3 = nn.Conv2d(in_channels=channels + 1, out_channels=self.fixed, kernel_size=3, stride=1,\n                                   padding=1, bias=True)\n\n    def forward(self, pan, ms_pro):  # ms_pro 6xhxw  pan 1xhxw\n        pan_guide = self.relu(self.conv1t6(pan))\n        pan_guide = self.conv6t6(pan_guide)  # 6*h*w\n        ms_pro = self.conv7t6_3(ms_pro)\n        # pan reshape -> 6xhw\n        # ms_pro reshape -> 6xhw  -> transpose > 6xwh\n        b, c, h, w = ms_pro.size()\n        pan_reshape = pan_guide.view(b, c, -1)  # 6xhw\n        ms_pro_transpose = ms_pro.permute(0, 1, 3, 2)\n        ms_pro_reshape = ms_pro_transpose.reshape(b, c, -1)  # 6xhw\n        # print(ms_pro_reshape.shape)\n        # ms使用了转置\n\n        # pan_matrix * ms_pro_matrix -> 6xhwxwh -> softmax ->6xhxw\n        pan_matrix = torch.mul(pan_reshape, ms_pro_reshape)  # b x c x hw x wh\n        # print(pan_matrix.shape)\n        pan_matrix = pan_matrix.view(b, c, h, w)\n        # print(pan_matrix.shape)\n        pan_matrix = pan_matrix.permute(0, 1, 3, 2)\n        pan_matrix = pan_matrix.contiguous().view(b, c, -1)\n       \n        pan_matrix = F.softmax(pan_matrix, dim=-1)\n        # print(pan_matrix.shape)\n\n        # transpose_R * ms_pro_matrix -> 6xhwxwh -> reshape -> output= 6xhxw\n        output = torch.mul(pan_reshape, pan_matrix)  # b x c x hw\n        output = output.contiguous().view(b, c, h, w)  # reshape to 6xhxw\n\n        # print(f"res{res.shape}")\n        return output\n\n\n\n\n\n\n\nclass PansharpeningNet(nn.Module):\n    def __init__(self, channels):\n        super(PansharpeningNet, self).__init__()\n\n        self.fixed = 4\n\n        # Downsampling and Upsampling layers\n        self.downsample1 = nn.Upsample(scale_factor=0.5, mode=\'bilinear\', align_corners=True)\n        self.downsample2 = nn.Upsample(scale_factor=0.25, mode=\'bilinear\', align_corners=True)\n        self.upsample1 = nn.Upsample(scale_factor=2, mode=\'bilinear\', align_corners=True)\n        self.upsample2 = nn.Upsample(scale_factor=4, mode=\'bilinear\', align_corners=True)\n        self.downsample200 = nn.Upsample(scale_factor=0.5, mode=\'bilinear\', align_corners=True)\n        # Feature extraction and fusion layers with individual names\n        self.prelu1 = nn.PReLU()\n        self.prelu2 = nn.PReLU()\n        self.prelu3 = nn.PReLU()\n\n        self.conv1t64 = nn.Conv2d(1, 64, kernel_size=3, padding=1)\n        self.conv64tnum = nn.Conv2d(64, channels, kernel_size=3, padding=1)\n        self.convnumt64 = nn.Conv2d(channels, 64, kernel_size=3, padding=1)\n\n        self.conv1t64_pan = nn.Conv2d(1, 64, kernel_size=3, padding=1)\n        self.conv64tnum_pan = nn.Conv2d(64, channels, kernel_size=3, padding=1)\n\n        self.convnum3tnum1 = nn.Conv2d(channels * 3, channels, kernel_size=3, padding=1)\n        self.convnum3tnum2 = nn.Conv2d(channels * 3, channels, kernel_size=3, padding=1)\n        self.convnum3tnum3 = nn.Conv2d(channels * 3, channels, kernel_size=3, padding=1)\n\n        self.conv48tnum1 = nn.Conv2d(in_channels=self.fixed * channels, out_channels=channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n        self.conv48tnum2 = nn.Conv2d(in_channels=self.fixed * channels, out_channels=channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n        self.conv48tnum3 = nn.Conv2d(in_channels=self.fixed * channels, out_channels=channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n\n        self.conv7t6_1 = nn.Conv2d(in_channels=channels + 1, out_channels=self.fixed, kernel_size=3, stride=1,\n                                   padding=1, bias=True)\n        self.conv7t6_2 = nn.Conv2d(in_channels=channels + 1, out_channels=self.fixed, kernel_size=3, stride=1,\n                                   padding=1, bias=True)\n        self.conv7t6_3 = nn.Conv2d(in_channels=channels + 1, out_channels=self.fixed, kernel_size=3, stride=1,\n                                   padding=1, bias=True)\n\n        self.conv2ctc1 = nn.Conv2d(in_channels=channels * 2, out_channels=channels, kernel_size=3, stride=1, padding=1,\n                                   bias=True)\n        self.conv2ctc2 = nn.Conv2d(in_channels=channels * 2, out_channels=channels, kernel_size=3, stride=1, padding=1,\n                                   bias=True)\n        self.conv2ctc3 = nn.Conv2d(in_channels=channels * 2, out_channels=channels, kernel_size=3, stride=1, padding=1,\n                                   bias=True)\n\n        self.SSA_blocks = nn.ModuleList([SSA(channels=channels) for _ in range(channels)])\n\n        self.SSA_blocks1 = nn.ModuleList([SSA(channels=channels) for _ in range(channels)])\n        self.SSA_blocks2 = nn.ModuleList([SSA(channels=channels) for _ in range(channels)])\n        self.conv1t1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=1, bias=True)\n        self.cov2t64 = nn.Conv2d(in_channels=2 + channels, out_channels=64, kernel_size=3, stride=1, padding=1,\n                                 bias=True)\n\n        self.res1 = Resblock()\n        self.res2 = Resblock()\n        self.res3 = Resblock()\n        self.res4 = Resblock()\n\n        self.backbone = nn.Sequential(\n                self.res1,\n                self.res2,\n                self.res3,\n                self.res4,\n            )\n        self.conv64t48 = nn.Conv2d(in_channels=64, out_channels=channels, kernel_size=3, stride=1, padding=1, bias=True)\n        self.fixed = 4;\n\n        # Downsampling and Upsampling layers\n        self.downsample = nn.Upsample(scale_factor=0.5, mode=\'bilinear\', align_corners=True)\n        self.upsample = nn.Upsample(scale_factor=2, mode=\'bilinear\', align_corners=True)\n        self.downsample1 = nn.Upsample(scale_factor=0.25, mode=\'bilinear\', align_corners=True)\n        self.upsample1 = nn.Upsample(scale_factor=4, mode=\'bilinear\', align_corners=True)\n        self.upsample100 = nn.Upsample(scale_factor=2, mode=\'bilinear\', align_corners=True)\n        self.downsample100 = nn.Upsample(scale_factor=0.5, mode=\'bilinear\', align_corners=True)\n        self.upsample101  = nn.Upsample(scale_factor=2, mode=\'bilinear\', align_corners=True)\n        self.upsample102\\\n            = nn.Upsample(scale_factor=2, mode=\'bilinear\', align_corners=True)\n        # Fusion layers\n        self.resblock = Resblock()\n        self.prelu = nn.PReLU()\n        self.conv1t64 = nn.Conv2d(1, 64, kernel_size=3, padding=1)\n        self.conv64tnum = nn.Conv2d(64, channels, kernel_size=3, padding=1)\n        self.convnumt64 = nn.Conv2d(channels, 64, kernel_size=3, padding=1)\n\n        self.conv1t64bf2 = nn.Conv2d(1, 64, kernel_size=3, padding=1)\n        self.conv64tnumbf2 = nn.Conv2d(64, channels, kernel_size=3, padding=1)\n        self.convnumt64bf2 = nn.Conv2d(channels, 64, kernel_size=3, padding=1)\n\n        self.convnumtnum = nn.Conv2d(channels * 3, channels * 3, kernel_size=3, padding=1)\n        self.convctc1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)  # convctc1\n        self.convctc2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)  # convctc2\n        self.convnum3tnum = nn.Conv2d(channels * 3, channels, kernel_size=3, padding=1)\n        self.relu = nn.ReLU()\n\n        self.conv2 = nn.Conv2d(in_channels=channels, out_channels=channels * 6, stride=1, kernel_size=3, padding=1,\n                               bias=True)\n\n        self.SSA_blocks = nn.ModuleList(\n            [SSA(channels=channels) for _ in range(channels)])  # Adding multi-scale blocks\n\n        self.conv48tnum1 = nn.Conv2d(in_channels=self.fixed * channels, out_channels=channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n        self.conv48tnum2 = nn.Conv2d(in_channels=self.fixed * channels, out_channels=channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n\n        self.conv48t48_1 = nn.Conv2d(in_channels=6 * channels, out_channels=6 * channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n        self.conv48t48_2 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=1, padding=1,\n                                     bias=True)\n        self.conv48t48_3 = nn.Conv2d(in_channels=6 * channels, out_channels=6 * channels, kernel_size=3, stride=1,\n                                     padding=1, bias=True)\n\n        self.conv7t6 = nn.Conv2d(in_channels=channels + 1, out_channels=self.fixed, kernel_size=3, stride=1, padding=1,\n                                 bias=True)\n\n        self.convnum3tn64 = nn.Conv2d(in_channels=channels * 3, out_channels=64, kernel_size=3, stride=1, padding=1,\n                                      bias=True)\n\n        self.group_size = 6 \n        self.channels = channels\n        self.cov2t64 = nn.Conv2d(in_channels=2 + channels, out_channels=64, kernel_size=3, stride=1, padding=1,\n                                 bias=True)\n        self.conv64t48 = nn.Conv2d(in_channels=64, out_channels=channels, kernel_size=3, stride=1, padding=1, bias=True)\n        self.conv1t1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=1, bias=True)\n\n        self.convnumt48 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, stride=1, padding=1,\n                                    bias=True)\n\n        self.relu = nn.ReLU(inplace=True)\n        self.conv8t4 = nn.Conv2d(in_channels=8, out_channels=channels, kernel_size=3, stride=1, padding=1,\n                                    bias=True)\n\n\n\n   \n    def forward(self, pan, lms):\n        # Stage 1: Downsample and upsample each input\n        pan_down_up = self.upsample1(self.downsample1(pan))\n        pan_updown = self.prelu(self.conv1t1(pan_down_up))\n        input = torch.cat([lms, pan, pan_updown], dim=1)\n\n        y = self.cov2t64(input)\n        y = self.backbone(y)\n        y1 = self.conv64t48(y)\n        rs = y1 + lms\n\n        # Process input channels separately\n        split_input = [rs[:, i:i + self.group_size, :, :] for i in range(0, 48, self.group_size)]\n        split_ms = [lms[:, i, :, :].unsqueeze(1) for i in range(self.channels)]\n\n        outputs = []\n        for input_channel, ms_channel, ssa in zip(split_input, split_ms, self.SSA_blocks):\n            x = torch.cat([rs, ms_channel], dim=1)\n\n            ms_one = ssa(pan, x)\n            outputs.append(ms_one)\n\n        pan_ms = torch.cat(outputs, dim=1)\n        pan_ms = self.prelu1(self.conv48tnum1(pan_ms))\n        fsm_msb = rs + pan_ms\n        res = fsm_msb\n\n        res_down = self.downsample100(res)\n\n        # Stage 2: MS upsample and next stage fusion\n        ms_up = self.upsample100(ms)\n        rs = res_down\n\n        split_input = [rs[:, i:i + self.group_size, :, :] for i in range(0, 48, self.group_size)]\n        split_ms = [ms_up[:, i, :, :].unsqueeze(1) for i in range(self.channels)]\n\n        outputs = []\n        for input_channel, ms_channel, ssa in zip(split_input, split_ms, self.SSA_blocks1):\n            x = torch.cat([rs, ms_channel], dim=1)\n\n            pan_down = self.downsample100(pan)\n            ms_one = ssa(pan_down, x)\n            outputs.append(ms_one)\n\n        pan_ms = torch.cat(outputs, dim=1)\n        pan_ms = self.prelu2(self.conv48tnum2(pan_ms))\n        fsm_msb = rs + pan_ms\n        res_down = fsm_msb\n\n        # Stage 3: Small scale stage\n        res_small = self.downsample200(res_down)\n        rs = res_small\n\n        split_input = [rs[:, i:i + self.group_size, :, :] for i in range(0, 48, self.group_size)]\n        split_ms = [ms[:, i, :, :].unsqueeze(1) for i in range(self.channels)]\n\n        outputs = []\n        for input_channel, ms_channel, ssa in zip(split_input, split_ms, self.SSA_blocks2):\n            x = torch.cat([rs, ms_channel], dim=1)\n\n            pan_down = self.downsample2(pan)\n            ms_one = ssa(pan_down, x)\n            outputs.append(ms_one)\n\n        pan_ms = torch.cat(outputs, dim=1)\n        pan_ms = self.prelu3(self.conv48tnum3(pan_ms))\n        res_small = pan_ms\n\n        # Output processing\n        ms_output = ms\n        ms_output = torch.cat([ms_output, res_small], dim=1)\n        ms_output = self.conv2ctc1(ms_output)\n\n        ms_output = self.upsample101(ms_output)\n        ms_output = torch.cat([ms_output, res_down], dim=1)\n        ms_output = self.conv2ctc2(ms_output)\n\n        ms_output = self.upsample102(ms_output)\n        ms_output = torch.cat([ms_output, res], dim=1)\n        ms_output = self.conv2ctc3(ms_output)\n\n\n\n\n        return ms_output\n', 'qb_profile.json': '{\n  "coverage": 17139,\n  "gnyq": [\n    0.34,\n    0.32,\n    0.3,\n    0.22\n  ],\n  "interpretation": "Fixed source-based MTF; geometry assigned only from original TRAIN GT/MS, never predictions or test data. Patch-interior agreement does not establish FR sensor physics.",\n  "kernel": "DLPan_MATLAB_radial_fwind1",\n  "kernel_size": 41,\n  "lr_margin": 5,\n  "max_sample_rmse_dn": 9.879355127045927e-13,\n  "normalized_mse": 1.1933945513453444e-32,\n  "operator_sha256": "88c38ec7f2c87487101ce71b8fdfbbb973e95d334bf07f24fa205f93c2765d2d",\n  "padding": "replicate",\n  "phase_candidates": [\n    [\n      2,\n      2\n    ],\n    [\n      1,\n      2\n    ],\n    [\n      2,\n      1\n    ]\n  ],\n  "phase_counts": {\n    "[1, 2]": 5671,\n    "[2, 1]": 5737,\n    "[2, 2]": 5731\n  },\n  "phase_policy": "fixed_per_training_sample",\n  "phases": [[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,1],[1,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,1],[1,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[2,1],[2,2],[1,2],[2,2],[2,1],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[1,2],[2,2],[2,2],[1,2],[2,2],[2,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[1,2],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[1,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,1],[1,2],[2,1],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[2,2],[2,2],[1,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,1],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[2,1],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[2,2],[2,2],[2,1],[2,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[1,2],[2,1],[2,1],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,2],[2,2],[1,2],[1,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[2,1],[1,2],[2,1],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,1],[2,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[1,2],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,1],[1,2],[2,1],[2,1],[2,2],[1,2],[2,2],[1,2],[2,1],[2,1],[2,2],[2,2],[2,1],[1,2],[2,1],[1,2],[1,2],[1,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[1,2],[1,2],[2,1],[2,1],[1,2],[2,2],[1,2],[1,2],[1,2],[1,2],[2,1],[1,2],[2,2],[2,2],[2,1],[2,1],[2,2],[2,2],[2,1],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[2,2],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[1,2],[2,2],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,2],[2,1],[2,2],[2,2],[1,2],[2,1],[2,2],[1,2],[1,2],[2,2],[2,1],[1,2],[1,2],[2,1],[1,2],[2,1],[2,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,2],[2,2],[2,1],[2,2],[2,2],[1,2],[2,2],[1,2],[1,2],[2,2],[2,1],[2,2],[1,2],[2,1],[2,2],[1,2],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,1],[2,2],[2,1],[2,1],[1,2],[2,1],[2,2],[2,2],[1,2],[2,2],[2,1],[1,2],[2,1],[2,2]],\n  "probes": [\n    {\n      "id": 0,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 3.005048864031219e-13,\n      "sha256": "91e9464e29f30d8a8e2b14a947afb2d814c47fe0a4c6d699f26eed0b19be832f"\n    },\n    {\n      "id": 552,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 1.8234283488323638e-13,\n      "sha256": "db9e15167ac3f446c9b34e8cc274622dd3902011152debf299346c5194feed27"\n    },\n    {\n      "id": 1105,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.368758862727127e-13,\n      "sha256": "952e0424d2dce63f823ca3a14eaaef048f2df57d7141bcdaaf336882cec97ec1"\n    },\n    {\n      "id": 1658,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.3812626088620963e-13,\n      "sha256": "078214688714abfa84291d8118560ee295a4b5c819967798fbb94ab502a42756"\n    },\n    {\n      "id": 2211,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.428700933498868e-13,\n      "sha256": "72fd9cacf187c9653d37796ef741684e9177be92fada3b9570c66908beeda8be"\n    },\n    {\n      "id": 2764,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 1.6521604142721986e-13,\n      "sha256": "1f58968203348f7fd9750798e3b2156522ad5a67d9961494e6a8e215314f25ed"\n    },\n    {\n      "id": 3317,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.706556245817075e-13,\n      "sha256": "4e10d48c25652f5f533878784314ae91b736172c0205b0c835202140b5beee46"\n    },\n    {\n      "id": 3869,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 1.4473160392538791e-13,\n      "sha256": "205d49cb26ead92d8a6afd231d3feb691bf37d94deec0c353801258b95e858da"\n    },\n    {\n      "id": 4422,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 2.1862263178906578e-13,\n      "sha256": "7f6c7e0b8125dea2f54a6849fb478aef5d4b9cf4f6bbd050b763ecf920e5f756"\n    },\n    {\n      "id": 4975,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.9534182630072859e-13,\n      "sha256": "09625b63b5ef53bf1fa45b391b625aa9885635dcb059278aa686dc336050ed59"\n    },\n    {\n      "id": 5528,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.3251563815288552e-13,\n      "sha256": "047dc11f3b54b119dfc398225c0f5fdb2ae97b0dca7d3793c2dd7a0ed645e8a7"\n    },\n    {\n      "id": 6081,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.125228547061468e-13,\n      "sha256": "a8be93cf02e431109c60434ddc277a61729377442e98b6e99c5a2c5a3c9f82a0"\n    },\n    {\n      "id": 6634,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.3310170136790175e-13,\n      "sha256": "d35f4dd7277834fa9a6ee76aa06923b28ace3cf47d39ee5a9f35b37ef60f26f0"\n    },\n    {\n      "id": 7186,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.4450371435926333e-13,\n      "sha256": "d826578acb9ead42609a435fcd29cb174d0811a864624932e29127cbf8b19e97"\n    },\n    {\n      "id": 7739,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.8381741184467031e-13,\n      "sha256": "f60f4b223d59601e8566d772ebc32f56df74d26f644559c2e726f83373d59909"\n    },\n    {\n      "id": 8292,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.8798634238907022e-13,\n      "sha256": "0cb3297b2fec943bd98418b9ebe3207a97adbaab4b745aa37901cd258b9799e9"\n    },\n    {\n      "id": 8845,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 1.8840458195335424e-13,\n      "sha256": "55a908e83742481ad0651c1c4e78201c1a44774bd20034a63112d9e16ef2fbcb"\n    },\n    {\n      "id": 9398,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 2.574185619541936e-13,\n      "sha256": "210f47f093059f63f76b9a37d3a034d199fcc257ead5534732417ef7c2a6f7b5"\n    },\n    {\n      "id": 9951,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 2.73528192112333e-13,\n      "sha256": "7328f8efbd2e976fbd8137b6d3e8fdf1dd4c0d2c094ab67d63ec0615f14bdb7c"\n    },\n    {\n      "id": 10503,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 2.533106721706053e-13,\n      "sha256": "024749cdde9d988ab2d68c6b4c5025353705d33d02e00d1b5d7319b30e89de99"\n    },\n    {\n      "id": 11056,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 3.2310168175564764e-13,\n      "sha256": "36abbb89a6277544802494e8867ab0bf3dbf86c429d43f0e0251a6d4bbb1a0df"\n    },\n    {\n      "id": 11609,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 2.7625886743069194e-13,\n      "sha256": "1460316de1749519346d13a73abe528ea99a0a1ba7db700a6b98a76fb8e9f47c"\n    },\n    {\n      "id": 12162,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 1.7837626258448088e-13,\n      "sha256": "6ccf182424361afca4a8b25e8b121b0471736aa664137e4673c1706d8645db11"\n    },\n    {\n      "id": 12715,\n      "phase": [\n        2,\n        1\n      ],\n      "rmse_dn": 2.362665894334605e-13,\n      "sha256": "c7064c9f8473d2422d1249654e34d7ec58c2ebda6e2b10884b791027e9c0128d"\n    },\n    {\n      "id": 13268,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 3.245719327316231e-13,\n      "sha256": "5d5364c9d28dd5032196a31d3af9d24429ceae5ff71db6365ec2ba1c7374bf52"\n    },\n    {\n      "id": 13820,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 3.1253785807176475e-13,\n      "sha256": "a0448f3c47a4f9df70016ec2ea64aa7d4d25b1a1a86aeee7feb0bcf4528b1882"\n    },\n    {\n      "id": 14373,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.4905763900028106e-13,\n      "sha256": "f6afe0206ec472149960802edae0f6396f5ff67be09ffcc06a5e36b2c0be7f98"\n    },\n    {\n      "id": 14926,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 2.4101205072074843e-13,\n      "sha256": "95a2163988f045aaa932197fe4788cd41bd4cf65ba5e8dfa1794cc0f05e4ef67"\n    },\n    {\n      "id": 15479,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 2.088056903542027e-13,\n      "sha256": "6bf2c530a82b9fa7b1279e9c95402f808bbe7588a3f6cf30d474d68e0902e4dc"\n    },\n    {\n      "id": 16032,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 1.2648636435432734e-13,\n      "sha256": "0d6082f6dd0083d8d06a1aec70f095e05e5c9be2ccc5ee4d3b784dcde96635e6"\n    },\n    {\n      "id": 16585,\n      "phase": [\n        1,\n        2\n      ],\n      "rmse_dn": 2.653089358764744e-13,\n      "sha256": "5b5fa4b5adf0d2eb8c0e4680f3c982d61a5f8bf580fbe0c39f823cd89f32827a"\n    },\n    {\n      "id": 17138,\n      "phase": [\n        2,\n        2\n      ],\n      "rmse_dn": 1.5524642887595108e-13,\n      "sha256": "2e31a882aa8ef98f7de35b8d2412955030a5f1df22713536e21fb03f694e6340"\n    }\n  ],\n  "ratio": 4,\n  "rmse_dn": 2.23619600876091e-13,\n  "sample_count": 17139,\n  "sampling": "decimate",\n  "sensor": "QB",\n  "source_commit": "a34f884af88967580e3ef4f61a89e12fdfe4cda5",\n  "status": "validated",\n  "tolerance": {\n    "max_sample_rmse_dn": 2.0,\n    "rmse_dn": 1.0\n  },\n  "train_path": "C:\\\\CtrS\\\\SSA-MRN\\\\data\\\\dataset\\\\QuickBird\\\\Training Dataset\\\\train_qb.h5",\n  "train_stat": {\n    "mtime_ns": 1790468226000000000,\n    "size": 5194901672\n  }\n}\n', 'SSA-MRN/src/ssamrn/__init__.py': '', 'SSA-MRN/src/ssamrn/models/__init__.py': ''}
WORKER_SOURCE = 'import os, sys, json, math, time, hashlib, signal, argparse, shutil\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\nSTOP = False\n\ndef stop_signal(*args):\n    global STOP\n    STOP = True\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(8 << 20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_json(value, path):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef cpu_tree(value):\n    if isinstance(value, torch.Tensor): return value.detach().cpu()\n    if isinstance(value, dict): return {k: cpu_tree(v) for k, v in value.items()}\n    if isinstance(value, list): return [cpu_tree(v) for v in value]\n    if isinstance(value, tuple): return tuple(cpu_tree(v) for v in value)\n    return value\n\n\ndef atomic_checkpoint(value, path):\n    path = Path(path)\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(cpu_tree(value), tmp)\n    with open(tmp, \'rb\') as f: os.fsync(f.fileno())\n    if path.exists(): os.replace(path, path.with_suffix(\'.previous.pt\'))\n    os.replace(tmp, path)\n\n\ndef load_checkpoint(folder):\n    for name in (\'latest.pt\', \'latest.previous.pt\'):\n        path = folder / name\n        if path.exists():\n            try: return torch.load(path, map_location=\'cpu\', weights_only=True)\n            except Exception as exc: print(f\'{name} 읽기 실패: {exc}\', flush=True)\n    if any(folder.glob(\'latest*pt\')): raise RuntimeError(\'복원 가능한 latest 가중치가 없습니다.\')\n    return None\n\n\ndef epoch_order(length, seed, epoch):\n    return torch.randperm(length, generator=torch.Generator().manual_seed(seed + epoch * 100003)).tolist()\n\n\ndef bank_choice(index, seed, epoch):\n    # Worker count and prefetch do not affect augmentation or shuffling on resume.\n    return ((int(index) * 2654435761 + seed * 2246822519 + epoch * 3266489917) & 0xffffffff) % 3\n\n\ndef kernels_for_factor(factor):\n    from ssamrn.observation import GNYQ, mtf_kernels\n    original = GNYQ[\'QB\']\n    try:\n        GNYQ[\'QB\'] = [v * factor for v in original]\n        return torch.from_numpy(mtf_kernels(\'QB\')).float()\n    finally:\n        GNYQ[\'QB\'] = original\n\n\n@torch.no_grad()\ndef observe(gt, kernel, phases):\n    # Exact source padding and phase; stride-4 convolution avoids full-HR output.\n    padded = F.pad(gt, (20, 20, 20, 20), mode=\'replicate\')\n    out = gt.new_empty(len(gt), 4, gt.shape[-2] // 4, gt.shape[-1] // 4)\n    for r, c in ((2,2), (1,2), (2,1)):\n        selected = (phases[:, 0] == r) & (phases[:, 1] == c)\n        if selected.any():\n            out[selected] = F.conv2d(padded[selected, :, r:, c:], kernel, stride=4, groups=4)\n    return out\n\n\n@torch.no_grad()\ndef make_delta(gt, kernel_difference, phases):\n    delta = observe(gt, kernel_difference, phases)\n    # 41-tap filter needs a 20-HR-pixel halo. Only the audited 6x6 LR interior changes.\n    mask = torch.zeros_like(delta)\n    mask[..., 5:-5, 5:-5] = 1\n    return delta * mask\n\n\ndef build_banks(cache, phases, config, device, report):\n    from ssamrn.models.interp23 import interp23tap\n    stamp = cache / \'mtf_bank_complete.json\'\n    if stamp.exists():\n        recorded = json.loads(stamp.read_text())\n        if recorded[\'fingerprint\'] not in config.get(\'accepted_fingerprints\',[config[\'fingerprint\']]): raise ValueError(\'MTF 캐시 설정 불일치\')\n        return recorded\n    gt = np.load(cache / \'train_gt.npy\', mmap_mode=\'r\')\n    ms = np.load(cache / \'train_ms.npy\', mmap_mode=\'r\')\n    nominal = kernels_for_factor(1.).to(device)\n    differences = [kernels_for_factor(f).to(device) - nominal for f in config[\'mtf_factors\']]\n    banks = []\n    for i in (1, 2):\n        dm = np.lib.format.open_memmap(cache / f\'delta_ms_{i}.npy\', mode=\'w+\', dtype=\'float32\', shape=ms.shape)\n        dl = np.lib.format.open_memmap(cache / f\'delta_lms_{i}.npy\', mode=\'w+\', dtype=\'float32\', shape=gt.shape)\n        banks.append((dm, dl))\n    rmse_sum, rmse_max, count = 0., 0., 0\n    for start in range(0, len(gt), 128):\n        end = min(start + 128, len(gt))\n        g = torch.from_numpy(np.array(gt[start:end])).to(device)\n        m = torch.from_numpy(np.array(ms[start:end])).to(device)\n        p = torch.tensor(phases[start:end], device=device)\n        errors = (observe(g, nominal, p)[..., 5:-5, 5:-5] - m[..., 5:-5, 5:-5]) * 2047\n        rmses = errors.square().mean((1,2,3)).sqrt()\n        rmse_sum += rmses.sum().item(); rmse_max = max(rmse_max, rmses.max().item()); count += len(g)\n        if not math.isfinite(rmse_max) or rmse_max > 2:\n            raise ValueError(\'원본 TRAIN의 MTF/위상 검증 실패: 2 DN 초과\')\n        for bank, difference in zip(banks, differences):\n            d = make_delta(g, difference, p)\n            l = interp23tap(d)\n            bank[0][start:end] = d.cpu().numpy()\n            bank[1][start:end] = l.cpu().numpy()\n        report(stage=\'MTF 캐시\', percent=100 * end / len(gt), batch=0, batches=0)\n        if STOP: raise InterruptedError(\'MTF 캐시 준비 중 종료 요청\')\n        if time.time() > config[\'deadline\']: raise TimeoutError(\'MTF 준비 중 세션 시간 제한\')\n    for dm, dl in banks: dm.flush(); dl.flush()\n    record = dict(fingerprint=config[\'fingerprint\'], samples=count, mean_rmse_dn=rmse_sum/count,\n                  max_rmse_dn=rmse_max, factors=config[\'mtf_factors\'], interior_lr_margin=5,\n                  policy=\'MS + (D_jitter-D_nominal)(GT) in audited interior; LMS + interp23(delta_MS)\')\n    atomic_json(record, stamp)\n    return record\n\n\nclass CachedPairs(Dataset):\n    def __init__(self, cache, split, variant, seed):\n        self.base = {k: np.load(cache / f\'{split}_{k}.npy\', mmap_mode=\'r\') for k in (\'gt\',\'pan\',\'lms\',\'ms\')}\n        self.variant, self.seed, self.epoch = variant, seed, 0\n        self.deltas = {}\n        if variant == \'mtf_aug\':\n            self.deltas = {i: {k: np.load(cache / f\'delta_{k}_{i}.npy\', mmap_mode=\'r\') for k in (\'lms\',\'ms\')} for i in (1,2)}\n    def __len__(self): return len(self.base[\'gt\'])\n    def __getitem__(self, token):\n        index, epoch = token if isinstance(token, tuple) else (token, 0)\n        sample = {k: np.array(v[index], copy=True) for k, v in self.base.items()}\n        bank = bank_choice(index, self.seed, epoch) if self.variant == \'mtf_aug\' else 0\n        if bank:\n            for k in (\'lms\', \'ms\'): sample[k] += self.deltas[bank][k][index]\n        return {k: torch.from_numpy(v) for k,v in sample.items()}\n\n\nclass ResidentPairs:\n    """Batch gathers and augmentation on device; no per-sample CPU copies or H2D."""\n    def __init__(self, source, device):\n        self.device, self.variant, self.seed = device, source.variant, source.seed\n        def upload(array):\n            out = torch.empty(array.shape, dtype=torch.float32, device=device)\n            for start in range(0, len(array), 256):\n                chunk = torch.from_numpy(np.array(array[start:start+256], copy=True))\n                out[start:start+len(chunk)].copy_(chunk)\n            return out\n        self.base = {k: upload(v) for k,v in source.base.items()}\n        self.deltas = {i: {k: upload(v) for k,v in values.items()} for i,values in source.deltas.items()}\n    def __len__(self): return len(self.base[\'gt\'])\n    def batch(self, indices, epoch=0):\n        ids = torch.as_tensor(indices, dtype=torch.long, device=self.device)\n        sample = {k: v.index_select(0,ids) for k,v in self.base.items()}\n        if self.variant == \'mtf_aug\':\n            choice = ((ids * 2654435761 + self.seed * 2246822519 + epoch * 3266489917) & 0xffffffff) % 3\n            for k in (\'lms\',\'ms\'):\n                # torch.where keeps the unmodified state bit-identical, including signed zero.\n                delta1 = self.deltas[1][k].index_select(0,ids)\n                delta2 = self.deltas[2][k].index_select(0,ids)\n                sample[k] = torch.where((choice==1)[:,None,None,None], sample[k]+delta1,\n                           torch.where((choice==2)[:,None,None,None], sample[k]+delta2, sample[k]))\n        return sample\n\n\nclass ResidentLoader:\n    def __init__(self, dataset, size, sampler=None):\n        self.dataset,self.size,self.sampler = dataset,size,sampler\n    def __iter__(self):\n        if self.sampler is not None:\n            for tokens in self.sampler:\n                indices = [t[0] for t in tokens]\n                yield self.dataset.batch(indices, self.sampler.epoch)\n        else:\n            for start in range(0,len(self.dataset),self.size):\n                yield self.dataset.batch(range(start,min(start+self.size,len(self.dataset))))\n    def __len__(self):\n        return len(self.sampler) if self.sampler is not None else math.ceil(len(self.dataset)/self.size)\n\n\ndef resident_bytes(*datasets):\n    return sum(v.nbytes for d in datasets for v in d.base.values()) + sum(v.nbytes for d in datasets for bank in d.deltas.values() for v in bank.values())\n\n\nclass ResumeBatches:\n    def __init__(self, dataset, size, seed):\n        self.dataset, self.size, self.seed = dataset, size, seed\n        self.epoch, self.start = 0, 0\n    def __iter__(self):\n        order = epoch_order(len(self.dataset), self.seed, self.epoch)\n        for start in range(self.start * self.size, len(order), self.size):\n            yield [(i, self.epoch) for i in order[start:start+self.size]]\n    def __len__(self): return math.ceil(len(self.dataset)/self.size) - self.start\n\n\n@torch.no_grad()\ndef validate(model, loader, device):\n    model.eval()\n    stats = torch.zeros(4, device=device, dtype=torch.float64)\n    images = 0\n    for b in loader:\n        pan, lms, ms, gt = [b[k].to(device, non_blocking=True) for k in (\'pan\',\'lms\',\'ms\',\'gt\')]\n        # All validation metrics computed from FP32 inference, regardless of AMP training.\n        pred = model(pan, lms, ms)\n        difference = pred - gt\n        stats[0] += difference.square().sum(); stats[1] += gt.numel()\n        stats[2] += (-10 * difference.square().mean((-2,-1)).clamp_min(1e-30).log10()).mean(1).sum()\n        norms = pred.norm(dim=1) * gt.norm(dim=1)\n        valid = norms > 0\n        angles = torch.rad2deg(torch.acos(((pred * gt).sum(1) / norms.clamp_min(1e-30)).clamp(-1,1)))\n        counts = valid.flatten(1).sum(1)\n        if (counts == 0).any(): raise ValueError(\'SAM 계산 불가\')\n        stats[3] += ((angles * valid).flatten(1).sum(1) / counts).sum()\n        images += len(gt)\n    s = stats.cpu().tolist()\n    if not all(math.isfinite(x) for x in s): raise RuntimeError(\'validation nonfinite\')\n    return s[0]/s[1], s[2]/images, s[3]/images\n\n\ndef train(config, variant, device=\'cuda:0\'):\n    from ssamrn.models.ssa_mrn import RestoredPansharpeningNet\n    folder = Path(config[\'output\']) / variant\n    folder.mkdir(parents=True, exist_ok=True)\n    cache = Path(config[\'cache\'])\n    torch.set_num_threads(2)\n    torch.manual_seed(config[\'seed\'])\n    if device.startswith(\'cuda\'): torch.cuda.manual_seed_all(config[\'seed\'])\n    torch.backends.cudnn.benchmark = not config[\'deterministic\']\n    torch.use_deterministic_algorithms(config[\'deterministic\'])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    epoch, next_batch, total, count, best, history, skipped = 0, 0, 0., 0, float(\'inf\'), [], 0\n    started = time.time()\n    def report(**updates):\n        record = dict(variant=variant, stage=\'학습\', epoch=epoch+1, target=config[\'epochs\'],\n                      elapsed_seconds=time.time()-started, updated=time.time())\n        record.update(updates)\n        atomic_json(record, folder / \'status.json\')\n    report(stage=\'준비\', percent=0)\n    if variant == \'mtf_aug\':\n        profile = json.loads((Path(config[\'code_root\']) / \'qb_profile.json\').read_text())\n        audit = build_banks(cache, profile[\'phases\'], config, device, report)\n        atomic_json(audit, folder / \'augmentation_audit.json\')\n    tr = CachedPairs(cache, \'train\', variant, config[\'seed\'])\n    va = CachedPairs(cache, \'val\', \'baseline\', config[\'seed\'])\n    sampler = ResumeBatches(tr, config[\'micro_batch\'], config[\'seed\'])\n    use_resident = False\n    if config.get(\'gpu_resident\',True) and device.startswith(\'cuda\'):\n        needed = resident_bytes(tr,va)\n        free,total_memory = torch.cuda.mem_get_info()\n        # Keep 3 GiB for model, activations, cuDNN workspaces and checkpoint operations.\n        use_resident = needed + (3 << 30) < free\n        print(f\'GPU cache: need {needed/(1<<30):.2f} GiB; free {free/(1<<30):.2f} GiB; enabled={use_resident}\',flush=True)\n        if use_resident:\n            report(stage=\'GPU 데이터 준비\', percent=0, gpu_cache_gib=needed/(1<<30))\n            tr_gpu,va_gpu = ResidentPairs(tr,device),ResidentPairs(va,device)\n            loader = ResidentLoader(tr_gpu,config[\'micro_batch\'],sampler)\n            validation = ResidentLoader(va_gpu,config[\'micro_batch\'])\n    if not use_resident:\n        options = dict(num_workers=config[\'workers\'], pin_memory=device.startswith(\'cuda\'))\n        if config[\'workers\']: options.update(persistent_workers=True, prefetch_factor=2)\n        loader = DataLoader(tr, batch_sampler=sampler, **options)\n        validation = DataLoader(va, batch_size=config[\'micro_batch\'], shuffle=False, **options)\n    model = RestoredPansharpeningNet(4, config[\'k\']).to(device)\n    initial_hash = hashlib.sha256(b\'\'.join(v.detach().cpu().numpy().tobytes() for v in model.state_dict().values())).hexdigest()\n    provenance = dict(config=config, initial_model_sha256=initial_hash, torch=str(torch.__version__),\n                      gpu=torch.cuda.get_device_name(0) if device.startswith(\'cuda\') else \'CPU\')\n    if not (folder/\'provenance.json\').exists(): atomic_json(provenance, folder/\'provenance.json\')\n    atomic_json(provenance, folder / (\'session_\' + time.strftime(\'%Y%m%d_%H%M%S\') + \'.json\'))\n    opt = torch.optim.Adam(model.parameters(), lr=config[\'lr\'])\n    amp = config[\'amp\'] and device.startswith(\'cuda\')\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=amp, init_scale=1024.)\n    ck = load_checkpoint(folder)\n    if ck:\n        if ck[\'fingerprint\'] not in config.get(\'accepted_fingerprints\',[config[\'fingerprint\']]) or ck[\'variant\'] != variant:\n            raise ValueError(\'코드/데이터/학습 조건이 달라 재개할 수 없습니다.\')\n        if ck[\'fingerprint\'] != config[\'fingerprint\']:\n            atomic_json(dict(previous_fingerprint=ck[\'fingerprint\'], current_fingerprint=config[\'fingerprint\'], policy=\'audited GPU-cache-only implementation migration\'),folder/\'resume_migration.json\')\n        model.load_state_dict(ck[\'model\']); opt.load_state_dict(ck[\'optimizer\']); scaler.load_state_dict(ck[\'scaler\'])\n        epoch, next_batch = ck[\'epoch\'], ck[\'next_batch\']\n        total, count, best, history, skipped = ck[\'total\'], ck[\'count\'], ck[\'best_mse\'], ck[\'history\'], ck[\'skipped_steps\']\n        torch.set_rng_state(ck[\'rng\'])\n        if amp or device.startswith(\'cuda\'): torch.cuda.set_rng_state(ck[\'cuda_rng\'], 0)\n        print(f\'{variant}: epoch {epoch+1}, batch {next_batch}부터 복원\', flush=True)\n    total = torch.tensor(total,dtype=torch.float64,device=device)\n    stepped = [False]\n    step_hook = opt.register_step_post_hook(lambda optimizer,args,kwargs: stepped.__setitem__(0,True))\n    def checkpoint(name=\'latest.pt\'):\n        value = dict(config=dict(config, sensor=\'QB\', variant=\'baseline\', training_variant=variant), fingerprint=config[\'fingerprint\'], variant=variant, epoch=epoch, next_batch=next_batch,\n                     best_mse=best, total=total.item(), count=count, history=history, skipped_steps=skipped,\n                     model=model.state_dict(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),\n                     rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state(0) if device.startswith(\'cuda\') else torch.empty(0,dtype=torch.uint8))\n        atomic_checkpoint(value, folder/name)\n        atomic_json(history, folder/\'history.json\')\n        (folder/\'history.jsonl\').write_text(\'\'.join(json.dumps(r)+\'\\n\' for r in history))\n        return value\n    stop_epoch = min(config[\'epochs\'], epoch + config[\'epochs_this_session\'])\n    batches = math.ceil(len(tr)/config[\'micro_batch\'])\n    accumulation = config[\'batch_size\']//config[\'micro_batch\']\n    last_save = time.time()\n    while epoch < stop_epoch:\n        model.train()\n        sampler.epoch, sampler.start = epoch, next_batch\n        tick = time.time()\n        opt.zero_grad(set_to_none=True)\n        for local_batch, b in enumerate(loader):\n            batch = sampler.start + local_batch\n            group_start = (batch//accumulation)*accumulation\n            # Account for the smaller final accumulation group without changing effective weighting.\n            group_samples = min(config[\'batch_size\'], len(tr) - group_start*config[\'micro_batch\'])\n            pan, lms, ms, gt = [b[k].to(device, non_blocking=True) for k in (\'pan\',\'lms\',\'ms\',\'gt\')]\n            with torch.autocast(device_type=device.split(\':\')[0], dtype=torch.float16, enabled=amp):\n                pred = model(pan,lms,ms)\n                loss = F.mse_loss(pred.float(), gt.float())\n            if not torch.isfinite(loss): raise RuntimeError(\'학습 loss nonfinite; 출력 보존 후 원인 확인 필요\')\n            scaler.scale(loss * (len(gt)/group_samples)).backward()\n            total += loss.detach().double()*len(gt); count += len(gt)\n            boundary = (batch+1)%accumulation == 0 or batch+1 == batches\n            if boundary:\n                stepped[0] = False\n                scaler.step(opt); scaler.update()\n                skipped += int(not stepped[0])\n                opt.zero_grad(set_to_none=True)\n                next_batch = batch+1\n                if next_batch % config[\'checkpoint_steps\'] == 0 or time.time()-last_save > 300:\n                    checkpoint(); last_save = time.time()\n                if STOP or time.time() >= config[\'deadline\'] or (config.get(\'test_stop_batch\') and next_batch >= config[\'test_stop_batch\']):\n                    checkpoint()\n                    report(stage=\'일시정지 · 저장 완료\', percent=100*next_batch/batches, batch=next_batch, batches=batches)\n                    return\n            if boundary and batch%10 < accumulation:\n                rate = (local_batch+1)/(time.time()-tick)\n                report(percent=100*(batch+1)/batches, batch=batch+1, batches=batches,\n                       train_mse=total.item()/count, eta_epoch_seconds=(batches-batch-1)/rate,\n                       best_val_mse=best if math.isfinite(best) else None,\n                       samples_per_second=rate*config[\'micro_batch\'], gpu_resident=use_resident,\n                       gpu_allocated_gib=torch.cuda.memory_allocated()/(1<<30) if device.startswith(\'cuda\') else 0.)\n        report(stage=\'검증\', percent=100, batch=batches, batches=batches)\n        val, psnr, sam = validate(model, validation, device)\n        improved = val < best\n        best = min(best, val)\n        row = dict(epoch=epoch+1, train_mse=total.item()/count, val_mse=val, val_psnr_band_mean_peak1=psnr,\n                   val_sam_deg=sam, seconds_this_segment=time.time()-tick, skipped_amp_steps=skipped)\n        history.append(row)\n        print(json.dumps(row), flush=True)\n        epoch += 1; next_batch = 0; total.zero_(); count = 0\n        if improved: checkpoint(\'best.pt\')\n        checkpoint(); last_save = time.time()\n        if STOP or time.time() >= config[\'deadline\']: break\n    report(stage=\'완료\' if epoch >= config[\'epochs\'] else \'세션 분량 완료 · 재개 가능\', percent=100, completed_epochs=epoch,\n           best_val_mse=best, next_batch=next_batch)\n    if epoch >= config[\'epochs\']:\n        atomic_json(dict(epochs=epoch, best_val_mse=best, fingerprint=config[\'fingerprint\']), folder/\'complete.json\')\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--config\', required=True); p.add_argument(\'--variant\', choices=[\'baseline\',\'mtf_aug\'], required=True)\n    args = p.parse_args()\n    config = json.loads(Path(args.config).read_text())\n    sys.path.insert(0, str(Path(config[\'code_root\'])/\'SSA-MRN/src\'))\n    signal.signal(signal.SIGTERM, stop_signal); signal.signal(signal.SIGINT, stop_signal)\n    try: train(config, args.variant)\n    except Exception as exc:\n        folder = Path(config[\'output\']) / args.variant\n        folder.mkdir(parents=True, exist_ok=True)\n        atomic_json(dict(stage=\'실패\', error=str(exc), updated=time.time()), folder/\'status.json\')\n        raise\n\nif __name__ == \'__main__\': main()\n'

BASE_COMMIT = 'be02b7cbf03fe00681194ae9fc9419661f24cd56'
RUN_NAME = f'{RUN_TAG}_qb_k{K}_s{SEED}' + ('_smoke' if SMOKE else '')
OUTPUT = Path('/kaggle/working') / RUN_NAME
CODE = Path('/tmp') / (RUN_NAME + '_code')
OUTPUT.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

# 같은 결과 폴더에서 중복 실행을 차단합니다.
import fcntl
lock = open(OUTPUT / '.run.lock', 'a')
try: fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
except BlockingIOError: raise RuntimeError('같은 실험이 이미 실행 중입니다. 기존 실행의 로그를 확인하세요.')
try:
    subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,utilization.gpu', '--format=csv'], check=True)
    used = [int(x) for x in subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True).splitlines()]
    if any(x > 1500 for x in used[:2]):
        raise RuntimeError('GPU에 다른 학습 또는 큰 텐서가 남아 있습니다. 기존 학습을 확인한 뒤 별도 세션에서 실행하세요.')
except FileNotFoundError: raise RuntimeError('NVIDIA GPU 세션이 아닙니다.')

for relative, content in EMBEDDED_SOURCES.items():
    target = CODE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
worker = CODE / 'pair_worker.py'
worker.write_text(WORKER_SOURCE, encoding='utf-8')
source_hashes = {p: hashlib.sha256(s.encode()).hexdigest() for p,s in EMBEDDED_SOURCES.items()}
source_hashes['pair_worker.py'] = hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()


def find_h5(tail):
    choices = [p for p in Path('/kaggle/input').rglob('*.h5')
               if (p.name == tail or p.name.endswith('__' + tail)) and '._' not in p.name and h5py.is_hdf5(p)]
    if len(choices) != 1: raise RuntimeError(f'{tail}: 유효한 H5가 {len(choices)}개입니다. 연결된 데이터셋을 확인하세요.')
    return choices[0]


def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 << 20), b''): h.update(block)
    return h.hexdigest()


TRAIN = find_h5('train_qb.h5')
VAL = find_h5('valid_qb.h5')
if TRAIN.resolve() == VAL.resolve(): raise ValueError('train/validation이 같습니다.')
print('TRAIN:', TRAIN, '\nVAL:', VAL)
data_info = {}
for role, path in (('train', TRAIN), ('val', VAL)):
    print(role, 'SHA-256 확인 중...', flush=True)
    sha = file_hash(path)
    manifest_path = path.parent/'path_manifest.json'
    if manifest_path.exists():
        entries = json.loads(manifest_path.read_text())['files']
        expected = [e for e in entries if e['upload_name'] == path.name]
        if len(expected) != 1 or expected[0]['sha256'] != sha:
            raise RuntimeError('업로드 manifest와 데이터 해시가 다릅니다.')
    with h5py.File(path,'r') as h:
        shapes = {k: list(h[k].shape) for k in ('gt','pan','lms','ms')}
        n,c,height,width = shapes['gt']
        assert c == 4 and height == 64 and width == 64, f'QB 64x64 학습 패치가 필요합니다: {shapes}'
        assert shapes['pan'] == [n,1,height,width] and shapes['lms'] == shapes['gt']
        assert shapes['ms'] == [n,4,16,16]
        data_info[role] = dict(sha256=sha, bytes=path.stat().st_size, shapes=shapes)
profile = json.loads(EMBEDDED_SOURCES['qb_profile.json'])
assert data_info['train']['shapes']['gt'][0] == profile['sample_count'], '검증된 QB TRAIN과 샘플 수가 다릅니다.'

conditions = dict(protocol='qb_mtf_interior_delta_v1', seed=SEED, k=K, epochs=TOTAL_EPOCHS,
                  batch_size=EFFECTIVE_BATCH, micro_batch=MICRO_BATCH, lr=1e-4, amp=AMP,
                  deterministic=DETERMINISTIC, smoke=SMOKE, mtf_factors=[.9,1.1],
                  data=data_info, code=source_hashes, torch=str(torch.__version__), cuda=str(torch.version.cuda))
fingerprint = hashlib.sha256(json.dumps(conditions,sort_keys=True).encode()).hexdigest()
# Only this reviewed original worker may migrate without changing scientific conditions.
LEGACY_WORKER_SHA256 = '6d920ca4dfc03fdda0228567eeab167888d799acf471477a03717ea6f222eb9d'
def compatible(previous):
    old = {k:previous.get(k) for k in conditions}
    if old == conditions: return True
    old_code = dict(old.get('code') or {})
    if old_code.get('pair_worker.py') != LEGACY_WORKER_SHA256: return False
    old_code['pair_worker.py'] = conditions['code']['pair_worker.py']
    old['code'] = old_code
    return old == conditions
legacy_conditions = dict(conditions, code=dict(conditions['code'], **{'pair_worker.py': LEGACY_WORKER_SHA256}))
accepted_fingerprints = [fingerprint, hashlib.sha256(json.dumps(legacy_conditions,sort_keys=True).encode()).hexdigest()]

CACHE = Path('/tmp') / ('ctrs_qb_cache_' + fingerprint[:16])
for legacy_fingerprint in accepted_fingerprints[1:]:
    legacy_cache = Path('/tmp') / ('ctrs_qb_cache_' + legacy_fingerprint[:16])
    stamp = legacy_cache/'data_complete.json'
    if stamp.exists() and json.loads(stamp.read_text()).get('fingerprint') == legacy_fingerprint:
        CACHE = legacy_cache
        print('호환되는 이전 데이터 캐시 재사용:',CACHE)
        break
CACHE.mkdir(parents=True, exist_ok=True)

# 이전 학습 출력에서 체크포인트를 복원합니다.
if RESUME_INPUT:
    candidate = Path(RESUME_INPUT)
    if not (candidate/'pair_config.json').is_file() and (candidate/RUN_NAME/'pair_config.json').is_file(): candidate /= RUN_NAME
    candidates = [candidate]
else:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('pair_config.json')
                  if compatible(json.loads(p.read_text()))]
if len(candidates) > 1: raise RuntimeError('재개 출력이 여러 개입니다. RESUME_INPUT에 사용할 폴더 하나를 지정하세요.')
if candidates and not (OUTPUT/'pair_config.json').exists():
    previous = candidates[0]
    if not compatible(json.loads((previous/'pair_config.json').read_text())):
        raise ValueError('재개 코드/데이터/조건 불일치')
    for variant in ('baseline','mtf_aug'):
        source = previous/variant
        if source.is_dir(): shutil.copytree(source, OUTPUT/variant, dirs_exist_ok=True)
    print('이전 출력 복원:', previous)
old_config = OUTPUT/'pair_config.json'
if old_config.exists():
    previous_config = json.loads(old_config.read_text())
    if not compatible(previous_config):
        raise ValueError('같은 RUN_NAME에 다른 실험이 있습니다. 새 SEED 또는 새 폴더를 사용하세요.')
    # Accepted legacy fingerprints are derived from exact matching conditions, never trusted from metadata.

# 두 프로세스가 공유하는 FP32 mmap 캐시. 원본 22 GB 전체 복사 없이 QB train/val만 사용.
cache_stamp = CACHE/'data_complete.json'
if not cache_stamp.exists():
    required = sum(np.prod(shape)*4 for info in data_info.values() for shape in info['shapes'].values())
    required += 2*(np.prod(data_info['train']['shapes']['lms'])+np.prod(data_info['train']['shapes']['ms']))*4
    if shutil.disk_usage(CACHE).free < required + (2 << 30): raise RuntimeError('캐시를 위한 디스크 공간이 부족합니다.')
    for role,path in (('train',TRAIN),('val',VAL)):
        with h5py.File(path,'r') as h:
            n = min(32,len(h['gt'])) if SMOKE else len(h['gt'])
            for key in ('gt','pan','lms','ms'):
                array = np.lib.format.open_memmap(CACHE/f'{role}_{key}.npy',mode='w+',dtype='float32',shape=(n,*h[key].shape[1:]))
                for start in range(0,n,128):
                    block = np.asarray(h[key][start:start+128],dtype='float32')/2047.
                    if not np.isfinite(block).all(): raise ValueError('데이터 nonfinite')
                    array[start:start+len(block)] = block
                array.flush(); del array
    cache_stamp.write_text(json.dumps(dict(fingerprint=fingerprint)))
    print('QB 데이터 캐시 생성 완료')
config = dict(conditions, fingerprint=fingerprint, accepted_fingerprints=list(set(accepted_fingerprints)), gpu_resident=GPU_RESIDENT, output=str(OUTPUT), cache=str(CACHE), code_root=str(CODE),
              epochs=1 if SMOKE else TOTAL_EPOCHS, workers=WORKERS_PER_GPU,
              checkpoint_steps=CHECKPOINT_STEPS, epochs_this_session=1 if SMOKE else EPOCHS_THIS_SESSION,
              deadline=time.time()+MAX_HOURS*3600, input_paths=dict(train=str(TRAIN),val=str(VAL)), base_commit=BASE_COMMIT)
old_config.write_text(json.dumps(config,ensure_ascii=False,indent=2))
shutil.copytree(CODE,OUTPUT/('source_snapshot_'+fingerprint[:16]),dirs_exist_ok=True)
processes = []
handles = []
status_display = display(HTML('두 GPU 학습을 준비합니다.'), display_id=True)
try:
    for gpu, variant in enumerate(('baseline','mtf_aug')):
        folder = OUTPUT/variant
        folder.mkdir(exist_ok=True)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
        log = open(folder/'train.log','a',buffering=1)
        handles.append(log)
        processes.append(subprocess.Popen([sys.executable,'-u',str(worker),'--config',str(old_config),'--variant',variant],env=env,stdout=log,stderr=subprocess.STDOUT))
    import html
    while any(p.poll() is None for p in processes):
        if any(p.poll() not in (None,0) for p in processes):
            for p in processes:
                if p.poll() is None: p.send_signal(signal.SIGTERM)
            for p in processes: p.wait()
            break
        rows = []
        for gpu,variant in enumerate(('baseline','mtf_aug')):
            status = OUTPUT/variant/'status.json'
            try: s = json.loads(status.read_text()) if status.exists() else {}
            except (ValueError,OSError): s = {}
            stage = html.escape(str(s.get('stage','준비')))
            error = html.escape(str(s.get('error','')))
            rows.append(f"<tr><td>GPU {gpu} · {variant}</td><td>{stage}</td><td>{s.get('epoch','-')}/{config['epochs']}</td><td>{s.get('percent',0):.1f}%</td><td>{s.get('batch','-')}/{s.get('batches','-')}</td><td>{s.get('samples_per_second',0):.1f}</td><td>{s.get('gpu_allocated_gib',0):.2f}</td><td>{error}</td></tr>")
        status_display.update(HTML('<table><tr><th>실험</th><th>상태</th><th>에폭</th><th>진행률</th><th>배치</th><th>패치/초</th><th>GPU GiB</th><th>오류</th></tr>'+''.join(rows)+'</table>'))
        time.sleep(5)
except KeyboardInterrupt:
    print('중단 요청: 두 프로세스가 optimizer step을 마치고 체크포인트를 저장합니다.',flush=True)
    for p in processes:
        if p.poll() is None: p.send_signal(signal.SIGTERM)
    for p in processes: p.wait()
finally:
    for p in processes:
        if p.poll() is None: p.send_signal(signal.SIGTERM)
    for p in processes:
        if p.poll() is None: p.wait()
    for handle in handles: handle.close()
    fcntl.flock(lock, fcntl.LOCK_UN); lock.close()

# 학습 결과와 실행 스냅샷만 저장. 캐시나 입력 데이터는 출력에 포함하지 않습니다.
if importlib.util.find_spec('matplotlib'):
    import matplotlib.pyplot as plt
    fig,axes = plt.subplots(1,4,figsize=(18,4))
    for variant in ('baseline','mtf_aug'):
        history_path = OUTPUT/variant/'history.json'
        if not history_path.exists(): continue
        history = json.loads(history_path.read_text())
        for ax,key in zip(axes,('train_mse','val_mse','val_psnr_band_mean_peak1','val_sam_deg')):
            ax.plot([r['epoch'] for r in history],[r[key] for r in history],label=variant)
            ax.set_title(key); ax.set_xlabel('Epoch'); ax.grid(alpha=.3)
            if history: ax.legend()
    fig.tight_layout(); fig.savefig(OUTPUT/'validation_curves.png',dpi=160); plt.close(fig)
files = {str(p.relative_to(OUTPUT)): dict(bytes=p.stat().st_size,sha256=file_hash(p)) for p in OUTPUT.rglob('*') if p.is_file() and p.name not in ('.run.lock','artifact_manifest.json') and not p.name.endswith('.tmp')}
(OUTPUT/'artifact_manifest.json').write_text(json.dumps(files,indent=2))
archive = OUTPUT.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and p.name != '.run.lock' and not p.name.endswith('.tmp'): z.write(p,str(p.relative_to(OUTPUT.parent)))
for variant,p in zip(('baseline','mtf_aug'),processes):
    print(variant, 'exit code:',p.returncode)
    status = OUTPUT/variant/'status.json'
    if status.exists(): print(status.read_text())
    if p.returncode:
        print('\n'.join((OUTPUT/variant/'train.log').read_text().splitlines()[-20:]))
print('결과:',OUTPUT,'\n체크포인트를 다음 세션에서 사용하려면 이 Notebook의 출력 저장 후 Add Input으로 연결하세요.')
display(FileLink(str(archive)))
if any(p.returncode for p in processes): raise RuntimeError('한 실험이 중단됐습니다. 위 로그를 확인하세요. 저장된 latest는 보존했습니다.')
